In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import pandas as pd

# Define the path to your CSV file
csv_file_path = '/content/drive/MyDrive/warehouse_messy_data.csv'

# Define dtypes to prevent premature conversion, especially for object columns
# that might contain mixed types or strings that look like numbers.
# Explicitly load these columns as strings to ensure they are handled by the
# cleaning logic, rather than being coerced to float by pd.read_csv.
dtype_spec = {
    'Product Name': str,
    'Category': str,
    'Warehouse': str,
    'Location': str,
    'Quantity': str, # This column is object but should be numeric; load as str to avoid issues
    'Supplier': str,
    'Status': str,
    'Last Restocked': str # This column is object but should be datetime; load as str to avoid issues
}

# Load the CSV file into a pandas DataFrame
try:
    # Use dtype_spec to ensure specific columns are loaded as strings
    df = pd.read_csv(csv_file_path, dtype=dtype_spec)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please ensure the path is correct.")
    df = None

if df is not None:
    # Display initial information about the DataFrame
    print("\nInitial DataFrame Info:")
    df.info()
    print("\nInitial DataFrame Head:")
    display(df.head())

Dataset loaded successfully.

Initial DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product ID      1000 non-null   int64  
 1   Product Name    1000 non-null   object 
 2   Category        1000 non-null   object 
 3   Warehouse       1000 non-null   object 
 4   Location        1000 non-null   object 
 5   Quantity        842 non-null    object 
 6   Price           793 non-null    float64
 7   Supplier        1000 non-null   object 
 8   Status          1000 non-null   object 
 9   Last Restocked  800 non-null    object 
dtypes: float64(1), int64(1), object(8)
memory usage: 78.3+ KB

Initial DataFrame Head:


/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
0,1102,gadget y,ELECTRONICS,Warehouse 2,Aisle 1,300,9.99,Supplier C,In Stock,NaN
1,1435,gadget y,ELECTRONICS,Warehouse 2,Aisle 4,two hundred,19.99,Supplier C,Out of Stock,NaN
2,1860,widget a,CLOTHING,Warehouse 2,Aisle 3,100,19.99,Supplier B,In Stock,20/12/2022
3,1270,gadget z,TOYS,Warehouse 2,Aisle 4,50,49.99,Supplier B,In Stock,20/12/2022
4,1106,widget a,FURNITURE,Warehouse 3,Aisle 3,two hundred,9.99,Supplier D,Out of Stock,25/04/2023


### Automatic Data Cleaning

Now, let's perform some automatic data cleaning steps:

1.  **Handle Missing Values**: We'll drop columns or rows that are entirely empty and try to fill numerical missing values with the mean/median or categorical ones with the mode. For this generic approach, we'll start by dropping rows where *all* values are missing.
2.  **Strip Whitespace**: Remove leading/trailing whitespace from string columns.
3.  **Attempt Type Conversion**: Try to convert columns to their most appropriate data types (e.g., numeric, datetime), coercing errors where necessary.
4.  **Drop Duplicate Rows**: Remove any exact duplicate rows that might exist in the dataset.

In [14]:
import pandas as pd # Ensure pandas is imported if this cell can be run independently

if df is not None:
    print("\nPerforming automatic data cleaning...")

    # 1. Drop rows where all elements are NaN
    initial_rows = df.shape[0]
    df.dropna(how='all', inplace=True)
    if df.shape[0] < initial_rows:
        print(f"Dropped {initial_rows - df.shape[0]} rows with all missing values.")

    # 2. Skip the separate whitespace stripping loop here. It will be handled within type conversion.

    # 3. Attempt Type Conversion for 'object' columns (before handling missing values)
    print("\n--- Starting Type Conversion Attempts ---")
    # Refresh the list of object columns before starting type conversion
    object_cols_before_conversion = df.select_dtypes(include=['object']).columns.tolist()
    print(f"Object columns identified before conversion: {object_cols_before_conversion}")

    for col in df.columns:
        # Only attempt conversion if the column is currently an 'object' type
        if df[col].dtype == 'object':
            print(f"Processing object column: '{col}'")
            original_non_null_count = df[col].notna().sum()
            print(f"  Original non-null count: {original_non_null_count}")

            if original_non_null_count == 0:
                print(f"  Column '{col}' is entirely null, skipping type conversion.")
                continue

            converted = False

            # Attempt datetime conversion first (dayfirst=True for DD/MM/YYYY)
            temp_dt = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
            dt_converted_count = temp_dt.notna().sum()
            dt_conversion_ratio = dt_converted_count / original_non_null_count if original_non_null_count > 0 else 0
            print(f"  Datetime conversion: {dt_converted_count} non-NaT values ({dt_conversion_ratio:.2f} ratio)")

            if dt_conversion_ratio > 0.8: # Heuristic: >80% non-null values convert to valid dates
                df[col] = temp_dt
                print(f"  --> Converted '{col}' to datetime (dayfirst=True) based on >80% valid date conversion.")
                converted = True

            if not converted:
                # Attempt numeric conversion
                temp_num = pd.to_numeric(df[col], errors='coerce')
                num_converted_count = temp_num.notna().sum()
                num_conversion_ratio = num_converted_count / original_non_null_count if original_non_null_count > 0 else 0
                print(f"  Numeric conversion: {num_converted_count} non-NaN values ({num_conversion_ratio:.2f} ratio)")

                if num_conversion_ratio > 0.8: # Heuristic: >80% non-null values convert to valid numbers
                    df[col] = temp_num
                    print(f"  --> Converted '{col}' to numeric based on >80% valid numeric conversion.")
                    converted = True

            # If still an object column (not datetime or numeric), now perform stripping and then consider category
            if not converted and df[col].dtype == 'object':
                # Apply strip only to string values in this column
                df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
                print(f"  Stripped whitespace from '{col}' (only for string values within object column).")

                # After stripping, check for category conversion
                nunique_val = df[col].nunique()
                row_count_fifth = df.shape[0] / 5
                print(f"  After stripping: Unique values: {nunique_val}, 1/5th of rows: {row_count_fifth:.2f}")

                if nunique_val < row_count_fifth and nunique_val > 1: # Only convert if more than 1 unique value
                    df[col] = df[col].astype('category')
                    print(f"  --> Converted '{col}' to category for memory efficiency.")
                else:
                    print(f"  --> Column '{col}' remains as object (string) due to low conversion rate or high cardinality.")

            print(f"  Final dtype of '{col}': {df[col].dtype}")
    print("--- Finished Type Conversion Attempts ---\n")

    # 4. Handle Missing Values (after type conversion, based on new dtypes)
    for col in df.columns:
        if df[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df[col]):
                mean_val = df[col].mean()
                formatted_mean_val = f"{mean_val:.2f}" if pd.notna(mean_val) else 'NaN'
                df[col] = df[col].fillna(mean_val if pd.notna(mean_val) else 0) # Use 0 if mean is NaN
                print(f"Filled missing numeric values in '{col}' with its mean ({formatted_mean_val}).")
            elif pd.api.types.is_datetime64_any_dtype(df[col]):
                mode_dt = df[col].mode()[0] if not df[col].mode().empty else pd.Timestamp('1970-01-01') # Default placeholder
                df[col] = df[col].fillna(mode_dt)
                print(f"Filled missing datetime values in '{col}' with its mode/placeholder ('{mode_dt}').")
            elif pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
                mode_val = df[col].mode()[0] if not df[col].mode().empty else 'Unknown' # Default placeholder
                df[col] = df[col].fillna(mode_val)
                print(f"Filled missing categorical/object values in '{col}' with its mode ('{mode_val}').")

    # 5. Drop Duplicate Rows
    initial_rows_duplicates = df.shape[0]
    df.drop_duplicates(inplace=True)
    if df.shape[0] < initial_rows_duplicates:
        print(f"Dropped {initial_rows_duplicates - df.shape[0]} duplicate rows.")

    print("\nData cleaning complete.")
    print("\nCleaned DataFrame Info:")
    df.info()
    print("\nCleaned DataFrame Head:")
    display(df.head())
else:
    print("Cannot perform cleaning as the DataFrame was not loaded.")


Performing automatic data cleaning...

--- Starting Type Conversion Attempts ---
Object columns identified before conversion: []
--- Finished Type Conversion Attempts ---


Data cleaning complete.

Cleaned DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 972 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product ID      972 non-null    int64  
 1   Product Name    972 non-null    float64
 2   Category        972 non-null    float64
 3   Warehouse       972 non-null    float64
 4   Location        972 non-null    float64
 5   Quantity        972 non-null    float64
 6   Price           972 non-null    float64
 7   Supplier        972 non-null    float64
 8   Status          972 non-null    float64
 9   Last Restocked  972 non-null    float64
dtypes: float64(9), int64(1)
memory usage: 83.5 KB

Cleaned DataFrame Head:


,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
0,1102,0.0,0.0,0.0,0.0,300.000000,9.99,0.0,0.0,0.0
1,1435,0.0,0.0,0.0,0.0,179.115479,19.99,0.0,0.0,0.0
2,1860,0.0,0.0,0.0,0.0,100.000000,19.99,0.0,0.0,0.0
3,1270,0.0,0.0,0.0,0.0,50.000000,49.99,0.0,0.0,0.0
4,1106,0.0,0.0,0.0,0.0,179.115479,9.99,0.0,0.0,0.0
